In [9]:
from transformers import pipeline
print('imported')
!hf auth login --token ""


imported
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `woopwoop` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `woopwoop`


In [ ]:
!pip install requests 
#!pip install transformers transformers 
#!pip install huggingface torch torchvision accelerate scikit-learn 
#!pip install -U huggingface_hub 
!pip install IPython
from IPython.display import clear_output
import time 
print('all installs good so far') 
print('View the terminal output')
print("=" * 50)
time.sleep(10)
clear_output()

In [ ]:
from PIL import Image
import time
import os
import random 
from typing import TypeVar 
from pathlib import Path
import cv2
import torch 
import requests 
import matplotlib.pyplot as plt 
from shutil import copy2
import yaml
from google.colab.patches import cv2_imshow
import numpy as np
T = TypeVar("T", str, Image.Image)

In [ ]:
from transformers import Sam3Processor, Sam3Model

from PIL import Image
import requests

# combine text + vision efficiency

model = Sam3Model.from_pretrained('facebook/sam3', device_map = 'cuda:0')
processor = Sam3Processor.from_pretrained('facebook/sam3')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stevenvafadar/camera-dataset")

print("Path to dataset files:", path)

In [ ]:
### FOR KAGGLE ONLY 
def load_images_from_folder(folder_path: str) -> list[Image.Image] | list[str]: 
    image_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    all_images = []

    for dirname, _, filenames in os.walk(folder_path):
        for filename in filenames:
            if filename.lower().endswith(image_extensions):
                image_path = os.path.join(dirname, filename)
                try: 
                    #image = Image.open(image_path).convert('RGB')
                    all_images.append(image_path)
                except Exception as error: 
                    print(f'Received Error: {error}')
                    continue
    return all_images 
    
start_time = time.time()
all_images = load_images_from_folder('/kaggle/input')
print(f'time taken: {time.time() - start_time}')
len(all_images)

In [ ]:
def copy_image(image_path: str, destination_folder: str) -> None: 
    source = Path(image_path)
    destination = Path(os.path.join(destination_folder, image_path))
    copy2(source, destination)
    

In [ ]:
def split_dataset(all_images = list[Image.Image] | list[str], train_split_percentage: int = 70, \
                  test_split_percentage : int = 30, valid_split_percentage: int = 10, \
                  use_validation: bool = False, seed: int = 42) -> dict[str, list[T]]: 
    if not all_images: 
        raise ValueError("Need to include images folder")

    shuffled_images = list(all_images)
    random_generator = random.Random(seed)
    random_generator.shuffle(shuffled_images)

    total_images = len(shuffled_images)

    train_count = int(total_images * train_split_percentage / 100)
    training_images = shuffled_images[:train_count] # first 70
    remaining_images = shuffled_images[train_count:] # last 30


    if use_validation: 
        test_split_percentage = 20 
        valid_count = int(total_images * valid_split_percentage / 100) # 1200
        print(f'valid_count: {valid_count}')
        test_count = int(total_images * test_split_percentage / 100)
        print(f'valid_count: {test_count}')

        validation_images = remaining_images[:valid_count] # 1800 images remain - first 1200 images
        testing_images = remaining_images[valid_count:] # last 600 images - starting at index 1200

        return {'train': training_images, 'valid': validation_images, 'test': testing_images}
        
        
    else: 
        validation_images = remaining_images 
        return {'train': training_images,'valid': testing_images}
        

In [ ]:
from transformers import pipeline
print('imported')
!hf auth login --token "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

In [ ]:
prompts = ['a coyote', 'a moose', 'a package', 'a deer']
classes_index = {int(i): str(label) for i, label in enumerate(prompts)}
classes_index

In [ ]:
HOME_DIR = os.path.join(os.getcwd())
HOME_DIR_IMAGES = os.path.join("/kaggle/input")
HOME_DIR, HOME_DIR_IMAGES

In [ ]:
def create_folders(folder_path: str, annotated_images: bool = True) -> None: 
    folders = {
        "train_images": os.path.join(folder_path, "train", "images"),
        "train_labels": os.path.join(folder_path, "train", "labels"),
        "valid_images": os.path.join(folder_path, "valid", "images"),
        "valid_labels": os.path.join(folder_path, "valid", "labels"),
    }
    if annotated_images: 
        folders['annotated_train_images'] = os.path.join(folder_path, 'annotated_images', 'train', 'images')
        folders['annotated_valid_images'] = os.path.join(folder_path, 'annotated_images', 'valid', 'images')


    for folder_path in folders.values(): 
        os.makedirs(folder_path, exist_ok = True)

# folder path splits into train and valid 

In [ ]:
# check to see if all folders got created and in the correct hierarchy
create_folders(HOME_DIR)
print(os.listdir(HOME_DIR))

In [ ]:
# %cd "valid"
# %cd train
os.listdir()

In [ ]:
### FOR LOCAL IMAGES ONLY
def process_images_in_batches(starting_image_index: str, images_dir: list[str], batch_size: int = 16) -> list[str]: 
    for i in range(starting_image_index, len(images_dir)):
        batch_images = images_dir[starting_image_index: starting_image_index + batch_size]
        ending_index = int(starting_batch_index + batch_size)
        return batch_images, ending_index

In [ ]:
def get_image_prefix(image_path: str) -> str: 
    image_prefix = image_path.split('/')[-1].split('.')[0]
    return image_prefix


In [ ]:
def write_bbox_txt(output_dir: str, bboxes: list[float], image_prefix: str, class_label: int, image_size: tuple = [int, int]) -> None: 
    if len(bboxes) > 0: 
        normalized_bboxes = normalize_bbox(bboxes, image_size = image_size)
        print(f'normalized bboxes: {normalized_bboxes}')
        print(f'writing out file to: {output_dir}/{image_prefix}.txt')
        with open(f'{output_dir}/labels/{image_prefix}.txt', 'a') as bbox_file: 
            bbox_file.write(f"{class_label} " + " ".join(f"{box}" for box in normalized_bboxes) + "\n")

In [ ]:
def write_image(output_dir: str, image_prefix: str) -> None: 
    pass

In [ ]:
def write_bbox_masks(masks: list[float], image_prefix: str, class_label: int, image_size: tuple = [int, int]) -> None: 
    if masks: 
        # line needs to be changed
        normalized_masks = normalize_bbox(bboxes, image_size[0], image_height[1])
        with open(f'{image_prefix}.txt') as bbox_file: 
            bbox_file.write(f"{class_label} " + " ".join(f"{box}" for box in bboxes))

In [ ]:
def normalize_bbox(bboxes: list[float], image_size: tuple = [int, int]) -> list[float]:
    x1, y1, x2, y2 = map(int, bboxes[0].tolist())
    image_height, image_width = image_size[0], image_size[1]
    normalized_x1 = (x1 / image_width)
    normalized_y1 = (y1 / image_height)
    normalized_x2 = (x2 / image_width)
    normalized_y2 = (y2 / image_height)

    return [normalized_x1, normalized_y1, normalized_x2, normalized_y2]

In [ ]:
def return_masks_scores_bboxes(results: list[dict], class_label: int) -> None: 
    bboxes = results['boxes']
    scores = results['scores']
    masks = results['masks']

    return bboxes, scores, masks

In [ ]:
def annotate_image_sam3(image, class_label: str, results: list[dict]) -> None: 
  image = Image.open(image)
  image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR) # not converted to BGR
  print(f'image shape after post processing: {image.shape}')
  bboxes = results['boxes']
  scores = results['scores']
  for (boxes, score) in zip(bboxes, scores):
    if score.item() > 0.4 :
      text_label = f'{class_label}: {score}%'
      print(type(boxes))
      x1, y1, x2, y2 = map(int, boxes.tolist())
      cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
      cv2.putText(image, text_label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
      cv2_imshow(image)


In [ ]:
def write_out_annotation(image, class_label: str, results: list[dict], color_indices: dict) -> None: 
    image_copy = image.copy()
    image = Image.open(image_copy)
    image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR) # convert back to original image
    bboxes = results['boxes']
    scores = results['scores']
    for (boxes, score) in zip(bboxes, scores):
     if score.item() > 0.4 :
       text_label = f'{class_label}: {score}%'
       print(type(boxes))
       x1, y1, x2, y2 = map(int, boxes.tolist())
       cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
       cv2.putText(image, text_label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
       cv2_imshow(image)

In [ ]:
def display_images(original_images: tuple[list[T],list[T]], annotated_images: tuple[list[T], list[T]]) -> None: 

    original_train, original_valid = original_images
    annotated_train, annotated_valid = annotated_images

    datasets = {
        "Train": (original_train, annotated_train),
        "Validation": (original_valid, annotated_valid),
    }

    for split_name, (originals, annotations) in datasets.items():
        if len(originals) != len(annotations):
            raise ValueError(
                f"{split_name} has {len(originals)} original images but "
                f"{len(annotations)} annotated images."
            )

        for index, (original, annotated) in enumerate(
            zip(originals, annotations)
        ):
            original_rgb = to_rgb_image(original)
            annotated_rgb = to_rgb_image(annotated)

            if notebook:
                plt.figure(figsize=(12, 5))

                plt.subplot(1, 2, 1)
                plt.imshow(original_rgb)
                plt.title(f"{split_name} original #{index}")
                plt.axis("off")

                plt.subplot(1, 2, 2)
                plt.imshow(annotated_rgb)
                plt.title(f"{split_name} annotated #{index}")
                plt.axis("off")

                plt.tight_layout()
                plt.show()

In [ ]:
def display_annotation_sam3(original_image: str, bbox: list[float]) -> None: 
    image = Image.open(original_image).convert("BGR")
    bbox = map(bbox, int)
    x1, y1, x2, y2 = bbox 
    cv2.rectangle(image, ())

In [ ]:


def generate_yaml_sam_3(class_labels: list[str], original_images_folder: str) -> None: 
    num_classes = len(class_labels)
    train_output_path = f"//{original_images_folder}/train/images"
    validation_output_Path = f"//{original_images_folder}/valid/images"
    data = {
        "names": class_labels,
        "nc": num_classes,
        "train": train_output_path,
        "val": validation_output_Path
    }
    with open(f"{original_images_folder}/data.yaml", "w") as f:
        yaml.safe_dump(
            data, 
            f, 
            sort_keys = False, 
            default_flow_style = False
        )

In [ ]:
def generate_auto_labelling_folder(all_images_folder: str, algorithm_type: str = "object_detection", ) -> None: 
    # obj detection format = class label bbox coordinates for each object in image
    pass
    

In [ ]:
def structure_autolabelling_and_yaml():
    # takes the results of the auto-labelling folder + the results of the yaml auto generation and combines it 
    # this should be the final annotation folder
    pass

In [ ]:
prompts = ['a coyote', 'a moose', 'a package', 'a deer']
def single_image_batch_inference(prompts: list[str], all_images: list[str], classes_index):
    #batch_images, ending_index = process_images_in_batches(all_images)
    dataset = split_dataset(all_images, 
                            train_split_percentage=70, 
                            valid_split_percentage = 10, 
                            test_split_percentage = 20,
                            use_validation=True,
                            seed = 42)
    print(f'classes index: {classes_index}')

    for folder_name, folder_split in dataset.items():
        print('folder_name', folder_name, 'folder_length', len(folder_split))
        # train, valid, images/labels 
        for image in folder_split: 
          image_prefix = get_image_prefix(image_path = image)
          print(f'image prefix: {image_prefix}')
          # need to split image into txt and train
          image_size = np.array(Image.open(image).convert('RGB')).shape # not necessary
          print(f'image shape before post processing: Height: {image_size[0]}\t{image_size[1]}')
          img_inputs = processor(images = image, return_tensors = 'pt').to(model.device)
          with torch.no_grad():
            vision_embeds = model.get_vision_features(pixel_values = img_inputs['pixel_values'])
        
          for class_label, class_name in classes_index.items():
            print(f'class label: {class_label}, class name: {class_name}')
            text_inputs = processor(text = class_name, return_tensors='pt').to(model.device)
            with torch.no_grad():
              outputs = model(vision_embeds=vision_embeds, **text_inputs)
        
            results = processor.post_process_instance_segmentation(
              outputs,
              threshold=0.5,
              mask_threshold=0.5,
              target_sizes=img_inputs.get("original_sizes").tolist())[0]
            if len(results['boxes']) > 0: 
              print(bboxes);
            annotate_image_sam3(image, class_label=class_name, results=results)
            print(f'image got annotated')
            bboxes, masks, scores = return_masks_scores_bboxes(results = results, class_label = class_label)
            
            output_dir = os.path.join(HOME_DIR, folder_name)
            write_bbox_txt(output_dir = output_dir, bboxes = bboxes, image_prefix = image_prefix, class_label = class_label, image_size = image_size)
        return

single_image_batch_inference(prompts = prompts, all_images = all_images, classes_index = classes_index)

In [46]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

def annotate_image(image, results: list[dict]) -> None:
  image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR) # not converted to BGR
  result = results[0]
  for (box, score, label) in zip(result['boxes'], result['scores'], result['labels']):
    box = [int(round(i, 2)) for i in box.tolist()]
    print(box)
    x1, y1, x2, y2 = box
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    text = f'{label}: {score}'
    cv2.putText(image, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
  cv2_imshow(image)


In [66]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import requests

# combine text + vision efficiency

model = Sam3Model.from_pretrained('facebook/sam3', device_map = 'cuda:0')
processor = Sam3Processor.from_pretrained('facebook/sam3')




model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

ValueError: .pt is not a valid TensorType, please select one of ['pt', 'np', 'mlx']

In [1]:
import yaml

def generate_yaml_sam_3(class_labels: list[str], original_images_folder: str) -> None: 
    num_classes = len(class_labels)
    train_output_path = f"//{original_images_folder}/train/images"
    validation_output_Path = f"//{original_images_folder}/valid/images"
    data = {
        "names": class_labels,
        "nc": num_classes,
        "train": train_output_path,
        "val": validation_output_Path
    }
    with open(f"{original_images_folder}/data.yaml", "w") as f:
        yaml.safe_dump(
            data, 
            f, 
            sort_keys = False, 
            default_flow_style = False
        )

In [ ]:
results

/usr/local/lib/python3.12/dist-packages/transformers/models/grounding_dino/processing_grounding_dino.py:91: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


[{'scores': tensor([0.4878, 0.4505, 0.4651], device='cuda:0'),
  'boxes': tensor([[344.5410,  23.1793, 637.3242, 374.5273],
          [ 12.2279,  52.0151, 316.8952, 472.6132],
          [ 38.7747,  70.0551, 176.6534, 118.0414]], device='cuda:0'),
  'text_labels': ['a cat', 'a cat', 'a remote control'],
  'labels': ['a cat', 'a cat', 'a remote control']}]

YOLO E TRAINING FILE FOLDER FORMAT 
SAME AS OTHER YOLO MODELS
SAM3 Masks -> CV2 find Contours -> results in a automated polygon file for yoloE training. Cannot do bbox -> polygon will 
produce rectangular (4) polygon values and not real (n) polygon values
hundreds of values = segmentation polygons not object detection values
sam3 mask shape (per one image) -> [1, 1150, 1500] (will be size of the image) | class: Tensor | 

In [ ]:
import yaml

def generate_yaml_sam_3(class_labels: list[str], original_images_folder: str) -> None: 
    num_classes = len(class_labels)
    train_output_path = f"//{original_images_folder}/train/images"
    validation_output_Path = f"//{original_images_folder}/valid/images"
    data = {
        "names": class_labels,
        "nc": num_classes,
        "train": train_output_path,
        "val": validation_output_Path
    }
    with open(f"{original_images_folder}/data.yaml", "w") as f:
        yaml.safe_dump(
            data, 
            f, 
            sort_keys = False, 
            default_flow_style = False
        )

In [ ]:
def generate_auto_labelling_folder(all_images_folder: str, algorithm_type: str = "object_detection", ) -> None: 
    # obj detection format = class label bbox coordinates for each object in image
    pass
    

In [ ]:
def structure_autolabelling_and_yaml():
    # takes the results of the auto-labelling folder + the results of the yaml auto generation and combines it 
    # this should be the final annotation folder
    pass